In [5]:
%%writefile room_manager.py
import os
import csv


class Room:
    def __init__(self, room_id, room_name, building, area_m2, lighting_kw, ac_kw, equipment_kw, hours_per_day):
        self.room_id = str(room_id).strip()
        self.room_name = str(room_name).strip()
        self.building = str(building).strip()
        self.area_m2 = float(area_m2)
        self.lighting_kw = float(lighting_kw)
        self.ac_kw = float(ac_kw)
        self.equipment_kw = float(equipment_kw)
        self.hours_per_day = float(hours_per_day)

        self.validate()

    def validate(self):
        if not self.room_id:
            raise ValueError("Room ID cannot be empty.")
        if self.area_m2 <= 0:
            raise ValueError("Room area must be greater than zero.")
        if any(p < 0 for p in [self.lighting_kw, self.ac_kw, self.equipment_kw]):
            raise ValueError("Power values must be nonnegative.")
        if not (0 <= self.hours_per_day <= 24):
            raise ValueError("Operating hours must be between 0 and 24.")

    def total_power(self):
        return self.lighting_kw + self.ac_kw + self.equipment_kw

    def daily_energy_consumption(self):
        return self.total_power() * self.hours_per_day

    def consumption_per_square_meter(self):
        return self.daily_energy_consumption() / self.area_m2

    def to_dict(self):
        return {
            "room_id": self.room_id,
            "room_name": self.room_name,
            "building": self.building,
            "area_m2": self.area_m2,
            "lighting_kw": self.lighting_kw,
            "ac_kw": self.ac_kw,
            "equipment_kw": self.equipment_kw,
            "hours_per_day": self.hours_per_day
        }


class RoomManager:
    FIELDNAMES = ["room_id", "room_name", "building", "area_m2",
                  "lighting_kw", "ac_kw", "equipment_kw", "hours_per_day"]
    TEXT_FIELDS = ("room_name", "building")      
    NUMBER_FIELDS = ("area_m2", "lighting_kw", "ac_kw", "equipment_kw", "hours_per_day")

    def __init__(self, filename="rooms.csv"):
        self.filename = filename
        self.rooms = {}         
        self.load_rooms()

    def load_rooms(self):
        self.rooms = {}
        if os.path.exists(self.filename):
            with open(self.filename, mode="r", encoding="utf-8", newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    try:
                        room = Room(
                            row["room_id"], row["room_name"], row["building"],
                            row["area_m2"], row["lighting_kw"], row["ac_kw"],
                            row["equipment_kw"], row["hours_per_day"]
                        )
                        self.rooms[room.room_id] = room
                    except (ValueError, KeyError, TypeError) as e:
                        print(f"Skipping invalid room: {e}")

    def save_rooms(self):
        with open(self.filename, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=self.FIELDNAMES)
            writer.writeheader()
            for room in self.rooms.values():
                writer.writerow(room.to_dict())

    def add_room(self, room):
        if room.room_id in self.rooms:
            raise ValueError(f"Room ID {room.room_id} already exists.")
        self.rooms[room.room_id] = room
        self.save_rooms()

    def list_rooms(self, limit=None):
        if not self.rooms:
            print("No rooms available.")
            return
        rooms = list(self.rooms.values())
        if limit is not None:
            rooms = rooms[:limit]           
        for room in rooms:
            print(f"ID: {room.room_id} | Name: {room.room_name} | Building: {room.building} | "
                  f"Daily Energy: {room.daily_energy_consumption():.2f} kWh")

    def search_room(self, keyword):
        return self.rooms.get(str(keyword).strip())

    def search_by_name_or_building(self, text):
        text = str(text).strip().lower()
        return [r for r in self.rooms.values()
                if text in r.room_name.lower() or text in r.building.lower()]

    def update_room(self, room_id, **kwargs):
        if room_id not in self.rooms:
            raise ValueError("Room ID not found.")
        room = self.rooms[room_id]
        backup = room.to_dict()               
        try:
            for key, value in kwargs.items():
                if key in self.TEXT_FIELDS:
                    setattr(room, key, str(value).strip())
                elif key in self.NUMBER_FIELDS:
                    setattr(room, key, float(value))
                else:
                    raise ValueError(f"Field '{key}' cannot be updated.")
            room.validate()
        except ValueError:
            for key, value in backup.items():  # rollback
                setattr(room, key, value)
            raise
        self.save_rooms()

   
    def total_daily_consumption(self):
        return sum(r.daily_energy_consumption() for r in self.rooms.values())

    def buildings(self):
        return {r.building for r in self.rooms.values()}

    def print_energy_report(self):
        if not self.rooms:
            print("No rooms available.")
            return
        print(f"{'ID':<5}{'Room':<18}{'Building':<16}{'kWh/day':>10}{'kWh/m2':>10}")
        for r in self.rooms.values():
            print(f"{r.room_id:<5}{r.room_name:<18}{r.building:<16}"
                  f"{r.daily_energy_consumption():>10.2f}{r.consumption_per_square_meter():>10.2f}")
        print(f"\nTotal daily consumption of all rooms: {self.total_daily_consumption():.2f} kWh")
        print(f"Buildings: {', '.join(sorted(self.buildings()))}")

Writing room_manager.py


## Tests: 3 valid cases + 3 invalid cases

In [6]:
import importlib, room_manager
importlib.reload(room_manager)           
from room_manager import Room

tests = [
    ("Valid 1: normal room",           "OK, 28.0 kWh/day", lambda: Room("T1","Test","Main",50,1.0,2.0,0.5,8)),
    ("Valid 2: 24-hour server room",   "OK, 316.8 kWh/day", lambda: Room("T2","Server","Main",20,0.2,5.0,8.0,24)),
    ("Valid 3: zero AC power",         "OK, 9.0 kWh/day", lambda: Room("T3","Store","Main",10,0.5,0,0.4,10)),
    ("Invalid 1: area = -10",          "ValueError", lambda: Room("T4","Bad","Main",-10,1,2,0.5,8)),
    ("Invalid 2: negative AC power",   "ValueError", lambda: Room("T5","Bad","Main",30,1,-2,0.5,8)),
    ("Invalid 3: 25 hours per day",    "ValueError", lambda: Room("T6","Bad","Main",30,1,2,0.5,25)),
]
print(f"{'Test':<32}{'Expected':<20}{'Actual':<45}Result")
for name, expected, build in tests:
    try:
        r = build()
        actual = f"OK, {r.daily_energy_consumption():.1f} kWh/day"
    except ValueError as e:
        actual = f"ValueError ({e})"
    ok = actual.startswith(expected.split(",")[0]) and (expected == "ValueError" or expected in actual)
    print(f"{name:<32}{expected:<20}{actual:<45}{'PASS' if ok else 'FAIL'}")

Test                            Expected            Actual                                       Result
Valid 1: normal room            OK, 28.0 kWh/day    OK, 28.0 kWh/day                             PASS
Valid 2: 24-hour server room    OK, 316.8 kWh/day   OK, 316.8 kWh/day                            PASS
Valid 3: zero AC power          OK, 9.0 kWh/day     OK, 9.0 kWh/day                              PASS
Invalid 1: area = -10           ValueError          ValueError (Room area must be greater than zero.)PASS
Invalid 2: negative AC power    ValueError          ValueError (Power values must be nonnegative.)PASS
Invalid 3: 25 hours per day     ValueError          ValueError (Operating hours must be between 0 and 24.)PASS


## Try the manager on the real file

In [7]:
from room_manager import RoomManager
manager = RoomManager("rooms.csv")
manager.list_rooms()
print()
manager.print_energy_report()

No rooms available.

No rooms available.
